In [1]:
from frameworks.LightenDiffusion.models.LightenDiffusion import Stage1
from frameworks.LightenDiffusion.models.decom import ImageEncoder, ImageDecoder, RetinexDecomposition
from dataset_registery.registery import DatasetManager
from torchsummary import summary
from frameworks.LightenDiffusion.training.stage1 import Stage1Trainer
from eda.helpers.data_helpers import split_dataloader
from eda.helpers.training_helpers import get_optimizer, get_scheduler
import torch
import os
%load_ext autoreload
%autoreload 2

In [2]:
data_name = "SICE_paired"
dataset_id = "okhater/SICE"
source = "huggingface"
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device used : {device}")

Device used : cuda


In [ ]:
registry = DatasetManager()
registry.initialize_dataset(
    name=data_name,
    dataset_id=dataset_id,
    splits=["train", "test"],
    dataset_type="paired",
    hf_cache_dir=f"../../datasets/{data_name}",
)

train_loader = registry.get_dataloader(data_name, "train", batch_size=16, shuffle=True)
test_loader = registry.get_dataloader(data_name, "test", batch_size=16, shuffle=True)
train_loader, val_loader = split_dataloader(train_loader, split_ratio=0.2)

loaders = {
    "train": train_loader, 
    "val": val_loader, 
    "test": test_loader
    }
for name, loader in loaders.items():
    total_samples = len(loader.dataset)
    print(f"Loader: {name}, Total samples: {total_samples}")
    for low_imgs, label in loader:
        print(f"Batch contains {len(low_imgs)} low image samples.")
        print("Low images batch shape:", low_imgs.shape)
        print("Label batch shape:", label.shape)
        break
    print("==="*20)

In [ ]:
latent_space_dim = 64
stage1_model = Stage1(
    encoder=ImageEncoder(latent_space_dim),
    decoder=ImageDecoder(latent_space_dim),
    decomposer=RetinexDecomposition(channels=latent_space_dim),
)
stage1_model.to(device)
summary(stage1_model, (2 , 3, 256, 256), batch_size=1)

In [ ]:
stage1_optimizer = get_optimizer(
    stage1_model,
    lr = 1e-4,
    weight_decay = 1e-4
    )
stage1_scheduler = get_scheduler(stage1_optimizer, gamma=.8)

In [ ]:
# setup = (weight_cont, weight_rec, weight_ref, weight_ill, lambda_g)
setup1 = (.1, 1, .1, .01, 10) # Paper values
setup2 = (.1, .8, .5, .01, 5) # More aggressive Consistency setup 
setup3 = (.5, .5, .1, .01, 10) # Balanced Reconstruction and Consistency
setup4 = (5, .1, 1, .01, 10) # debug
setup = setup1
num_epochs = 200
val_frequency = num_epochs // 5
patience = val_frequency * 2

In [ ]:
weight_cont, weight_rec, weight_ref, weight_ill, lambda_g = setup
trainer1 = Stage1Trainer(
    model=stage1_model,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=stage1_optimizer,
    device=device,
    val_frequency = val_frequency,
    patience = patience,
    num_epochs=num_epochs,
    scheduler=stage1_scheduler,
    weight_cont = weight_cont,
    weight_rec = weight_rec,
    weight_ref = weight_ref,
    weight_ill = weight_ill,
    lambda_g = lambda_g,
    num_visualizations=1
)
best_stage1, metrics1 = trainer1.train()

In [ ]:
directory = "/home/grads/o/omarkhater/projects/lle-generative-priors/frameworks/LightenDiffusion/trained_models/stage1/"
file_path = os.path.join(directory, "stage1_beta2.pth")
if not os.path.exists(directory):
    os.makedirs(directory)
torch.save(best_stage1.state_dict(), file_path)